# Airbnb Paris – Exp 4: ConTextTab vs. TabPFN-Klassifikation
- Binäre Klassifikation des ICC-Labels `is_top_rating` (Inlier = rating == 5 → 1, Outlier = rating <= 3 → 0)
- Zwei Verteilungs-Varianten je Modell: `original` (natürliche Rate, Train 4000 / Test 3000) und `balanced_1to4` (Train 120/480, Test 30/120, seed 42)
- ConTextTab: numerisch + Freitext (nativ); TabPFN: nur numerisch
- Pro Modell & Variante: Average Precision, AUPRC, AUC-ROC, Classification Report; `distribution` als MLflow-Param

In [ ]:
import torch  # vor sap_rpt_oss laden (TORCH_LIBRARY-Doppelregistrierung vermeiden)
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, precision_recall_curve, auc
from tabpfn import TabPFNClassifier
from sap_rpt_oss import SAP_RPT_OSS_Classifier

## Setup & MLflow
- `.env` laden; Tracking nach `../../mlruns`, Experiment `airbnb_paris_experiment_4`

In [2]:
SEED = 42
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_4")

/home/debian/foundational_tabular_od/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/airbnb_notebooks/exp4/../../mlruns/902320835569643670', creation_time=1781194963834, experiment_id='902320835569643670', last_update_time=1781194963834, lifecycle_stage='active', name='airbnb_paris_experiment_4', tags={}, trace_location=None, workspace='default'>

## Daten laden
- `cleaned_text_airbnb_paris.csv` (numerisch/encoded + 4 Freitext, `row_id`-Index); leere Texte als ""
- `X` = numerisch + Freitext (ConTextTab); `Xnum` = nur numerisch (TabPFN)

In [3]:
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]
LABEL = "is_top_rating"

df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
X = df.drop(columns=[LABEL])      # numerisch + Freitext
Xnum = X.drop(columns=TEXT_COLS)   # nur numerisch
y = df[LABEL]
print(df.shape, "| Verteilung:", y.value_counts().to_dict())

(18350, 45) | Verteilung: {1: 17629, 0: 721}


## Gemeinsames Subsampling – zwei Varianten
- Outlier = Minderheitsklasse (`is_top_rating == 0`, rating <= 3). `original`: Originalverteilung, Train-Kontext 4000 (TabPFN-Limit), Test 3000 (disjunkt). `balanced_1to4`: Train 120/480, Test 30/120
- Identische Zeilen für beide Modelle → fairer Vergleich; seed 42

In [ ]:
outlier_label = y.value_counts().idxmin()
inlier_label = y.value_counts().idxmax()
rng = np.random.RandomState(SEED)
idx_out = df.index[y == outlier_label].to_numpy()
idx_in = df.index[y == inlier_label].to_numpy()
rng.shuffle(idx_out)
rng.shuffle(idx_in)
rate = len(idx_out) / len(df)   # natürliche Outlier-Rate

# Variante "original": Originalverteilung, Train-Kontext auf 4000 gedeckelt (TabPFN-Limit), großer Test
N_CTX, N_TEST = 4000, 3000
o_ctx, o_te = round(N_CTX * rate), round(N_TEST * rate)
# Variante "balanced_1to4": künstlich balanciert wie bisher
N_TRAIN_OUT, N_TRAIN_IN, N_TEST_OUT, N_TEST_IN = 120, 480, 30, 120

splits = {
    "original": (np.concatenate([idx_out[:o_ctx], idx_in[:N_CTX - o_ctx]]),
                 np.concatenate([idx_out[o_ctx:o_ctx + o_te], idx_in[N_CTX - o_ctx:N_CTX - o_ctx + N_TEST - o_te]])),
    "balanced_1to4": (np.concatenate([idx_out[:N_TRAIN_OUT], idx_in[:N_TRAIN_IN]]),
                      np.concatenate([idx_out[N_TRAIN_OUT:N_TRAIN_OUT + N_TEST_OUT], idx_in[N_TRAIN_IN:N_TRAIN_IN + N_TEST_IN]])),
}
for k, (tr, te) in splits.items():
    rng.shuffle(tr)
    rng.shuffle(te)
    print(f"{k:14s} Train {len(tr)} (Outlier {int((y.loc[tr] == outlier_label).sum())}) | "
          f"Test {len(te)} (Outlier {int((y.loc[te] == outlier_label).sum())})")

## ConTextTab (numerisch + Freitext)
- SAP-rpt-1-oss, max_context_size=8192, bagging=8; Score = P(Outlier)

In [ ]:
for setting, (train_idx, test_idx) in splits.items():
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]
    y_bin = (y_test == outlier_label).astype(int)
    outlier_col = sorted(np.unique(y_train).tolist()).index(outlier_label)

    clf = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)
    t0 = time.time()
    clf.fit(X.loc[train_idx], y_train)
    proba = clf.predict_proba(X.loc[test_idx])[:, outlier_col]
    pred = clf.predict(X.loc[test_idx])
    runtime = time.time() - t0

    ap = average_precision_score(y_bin, proba)
    prec, rec, _ = precision_recall_curve(y_bin, proba)
    auprc = auc(rec, prec)
    auroc = roc_auc_score(y_bin, proba)
    print(f"ConTextTab [{setting}] – AP={ap:.4f}  AUPRC={auprc:.4f}  AUC-ROC={auroc:.4f}  t={runtime:.1f}s")
    print(classification_report(y_test, pred, digits=4, zero_division=0))

    with mlflow.start_run(run_name=f"contexttab_{setting}"):
        mlflow.log_params({"distribution": setting, "n_train": len(train_idx), "n_test": len(test_idx),
                           "n_features": X.shape[1], "max_context_size": 8192, "bagging": 8, "random_state": SEED})
        mlflow.log_metrics({"average_precision": float(ap), "auprc": float(auprc), "auc_roc": float(auroc), "runtime_s": round(runtime, 2)})

## TabPFN (nur numerisch)
- Freitexte gedroppt; Score = P(Outlier)

In [ ]:
for setting, (train_idx, test_idx) in splits.items():
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]
    y_bin = (y_test == outlier_label).astype(int)
    outlier_col = sorted(np.unique(y_train).tolist()).index(outlier_label)

    tab = TabPFNClassifier()
    t0 = time.time()
    tab.fit(Xnum.loc[train_idx].values, y_train.values)
    proba = tab.predict_proba(Xnum.loc[test_idx].values)[:, outlier_col]
    pred = tab.predict(Xnum.loc[test_idx].values)
    runtime = time.time() - t0

    ap = average_precision_score(y_bin, proba)
    prec, rec, _ = precision_recall_curve(y_bin, proba)
    auprc = auc(rec, prec)
    auroc = roc_auc_score(y_bin, proba)
    print(f"TabPFN [{setting}] – AP={ap:.4f}  AUPRC={auprc:.4f}  AUC-ROC={auroc:.4f}  t={runtime:.1f}s")
    print(classification_report(y_test, pred, digits=4, zero_division=0))

    with mlflow.start_run(run_name=f"tabpfn_classification_{setting}"):
        mlflow.log_params({"distribution": setting, "n_train": len(train_idx), "n_test": len(test_idx),
                           "n_features": Xnum.shape[1], "random_state": SEED})
        mlflow.log_metrics({"average_precision": float(ap), "auprc": float(auprc), "auc_roc": float(auroc), "runtime_s": round(runtime, 2)})